# NOAA NCEI — Official Station Weather Records (GHCN-Daily)

**What it is.** The authoritative U.S. climate archive. Two access paths:
- **NCEI Access Data Service** — **token-free**, great for pulling a known station's
  daily summaries.
- **CDO API v2** — needs a **free token**; best for *discovering* stations by area and for
  rich metadata.

| | |
|---|---|
| **Coverage** | Global GHCN-Daily; dense U.S. station network |
| **Cost / key** | Access service: **no key** · CDO API: **free token** |
| **Token** | https://www.ncdc.noaa.gov/cdo-web/token  → put in `.env` as `NOAA_CDO_TOKEN` |
| **Docs** | https://www.ncei.noaa.gov/support/access-data-service-api-user-documentation |

**Why it's in Tracera.** Official, QA'd station observations — the record of *what actually
happened* at a place. For fast modeling history prefer **gridMET / Open-Meteo / NASA POWER**
(all no-key and already wired); reach for NOAA when you need the official station of record.

> NCEI services can be slow/unavailable; the calls below use short timeouts and fail
> gracefully rather than hang.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### Path A — NCEI Access Data Service (no token). Daily summaries for a known station.

In [2]:
NCEI = "https://www.ncei.noaa.gov/access/services/data/v1"

def ncei_daily(station, start, end, datatypes=("TMAX", "TMIN", "PRCP"), timeout=25):
    """GHCN-Daily summaries (metric). station e.g. 'USW00014933' (Des Moines Intl)."""
    try:
        r = requests.get(NCEI, params={"dataset": "daily-summaries", "stations": station,
            "startDate": start, "endDate": end, "dataTypes": ",".join(datatypes),
            "format": "json", "units": "metric"}, timeout=timeout)
        if r.status_code == 200 and r.text.strip().startswith("["):
            df = pd.DataFrame(r.json())
            for c in datatypes:
                if c in df: df[c] = pd.to_numeric(df[c], errors="coerce")
            return df
        print("NCEI returned", r.status_code, "- try again shortly.")
    except requests.exceptions.RequestException as e:
        print(f"NCEI slow/unreachable ({type(e).__name__}). Rerun later; the request code is ready.")
    return pd.DataFrame()

noaa = ncei_daily("USW00014933", "2023-06-01", "2023-06-07")   # Des Moines International
noaa if not noaa.empty else "No data this run (NCEI transient)."

,DATE,STATION,TMAX,TMIN,PRCP
0,2023-06-01,USW00014933,28.3,18.9,25.7
1,2023-06-02,USW00014933,30.6,19.4,11.4
2,2023-06-03,USW00014933,31.7,18.3,0.0
3,2023-06-04,USW00014933,31.1,19.4,0.0
4,2023-06-05,USW00014933,29.4,18.3,0.0
5,2023-06-06,USW00014933,31.7,17.2,0.0
6,2023-06-07,USW00014933,26.7,18.9,0.0


### Path B — CDO API v2 (needs `NOAA_CDO_TOKEN`). Discover stations near the field.

In [3]:
CDO = "https://www.ncei.noaa.gov/cdo-web/api/v2"
TOKEN = get_key("NOAA_CDO_TOKEN", required=False)

def cdo_stations_near(lat=LAT, lon=LON, d=0.4, timeout=25):
    if not TOKEN:
        print("Add NOAA_CDO_TOKEN to .env to use the CDO API. Get one free:")
        print("  https://www.ncdc.noaa.gov/cdo-web/token")
        return pd.DataFrame()
    extent = f"{lat-d},{lon-d},{lat+d},{lon+d}"
    try:
        r = requests.get(f"{CDO}/stations", headers={"token": TOKEN},
            params={"extent": extent, "datasetid": "GHCND", "limit": 10}, timeout=timeout)
        return pd.DataFrame(r.json().get("results", []))
    except requests.exceptions.RequestException as e:
        print(f"CDO slow/unreachable ({type(e).__name__}).")
        return pd.DataFrame()

stns = cdo_stations_near()
stns[["id", "name", "mindate", "maxdate"]] if not stns.empty else "Provide a token to list stations."

,id,name,mindate,maxdate
0,GHCND:US1IABN0001,"BOONE 5.7 ESE, IA US",2007-07-27,2009-04-29
1,GHCND:US1IABN0009,"MADRID 0.4 SE, IA US",2007-08-26,2023-02-27
2,GHCND:US1IABN0013,"MADRID 5.8 NNW, IA US",2017-04-11,2026-07-08
3,GHCND:US1IABN0019,"BOONE 1.1 N, IA US",2024-03-26,2026-05-17
4,GHCND:US1IABN0020,"MADRID 0.5 NE, IA US",2024-05-31,2026-07-08
5,GHCND:US1IADL0041,"MADRID 3.3 SSW, IA US",2024-08-04,2026-07-05
6,GHCND:US1IAMS0004,"STATE CENTER 0.3 WNW, IA US",2007-08-20,2008-06-25
7,GHCND:US1IAPK0002,"ANKENY 2.8 SW, IA US",2008-01-23,2015-11-12
8,GHCND:US1IAPK0012,"GRIMES 3.8 N, IA US",2002-04-21,2010-08-25
9,GHCND:US1IAPK0014,"ANKENY 2.3 WNW, IA US",2007-08-09,2021-11-05


### Notes & how to extend
- **Station IDs:** the Access service wants a bare GHCN id (`USW00014933`); the CDO API uses
  `GHCND:USW00014933`. Discover ids with Path B, then pull history fast with Path A.
- **Datatypes:** `TMAX`, `TMIN`, `PRCP`, `SNOW`, `SNWD`, `AWND` (wind), plus derived
  `daily-summaries` fields. Add `units=standard` for °F/inches.
- **Bulk:** for whole-network history, NCEI publishes GHCN-Daily as flat files on its FTP/HTTPS
  server — faster than the API for large pulls.
- For routine modeling, the no-key **gridMET / Open-Meteo / NASA POWER** notebooks are faster
  and gap-free; use NOAA for the official station record.